# Notebook 3 — Qwen Parser (LLM Fallback)

Qwen handles the 10-15% of inputs that the rule-based parser couldn't classify.

**Exit criteria:** Qwen correctly classifies 80%+ of ambiguous cases with valid JSON output.

> **Note:** This notebook requires Qwen model to be available locally. If not set up yet, use the mock mode to validate the prompt and flow.

## 1. Setup — Try Loading Qwen

In [1]:

import json
import re
import subprocess
import sys

QWEN_AVAILABLE = False

# Try llama-cpp-python first
try:
    from llama_cpp import Llama
    QWEN_AVAILABLE = True
    USE_BACKEND = 'llama_cpp'
    print("llama-cpp-python available")
except ImportError:
    print("llama-cpp-python not installed")

# Try transformers if llama_cpp not available
if not QWEN_AVAILABLE:
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM
        import torch
        QWEN_AVAILABLE = True
        USE_BACKEND = 'transformers'
        print("transformers available")
    except ImportError:
        print("transformers not installed")

if not QWEN_AVAILABLE:
    print("\nNo LLM backend found. Running in MOCK MODE.")
    print("Install one of:")
    print("  pip install llama-cpp-python")
    print("  pip install transformers torch")
    USE_BACKEND = 'mock'


llama-cpp-python available


## 2. Load Model

In [2]:

llm = None

if USE_BACKEND == 'llama_cpp':
    # Update this path to your GGUF model file
    MODEL_PATH = "./models/qwen2.5-4b-instruct-q4_k_m.gguf"
    
    import os
    if os.path.exists(MODEL_PATH):
        llm = Llama(
            model_path=MODEL_PATH,
            n_ctx=2048,
            n_threads=4,
            verbose=False
        )
        print(f"Model loaded from {MODEL_PATH}")
    else:
        print(f"Model file not found: {MODEL_PATH}")
        print("Download from: https://huggingface.co/Qwen/Qwen2.5-4B-Instruct-GGUF")
        print("Falling back to MOCK MODE")
        USE_BACKEND = 'mock'

elif USE_BACKEND == 'transformers':
    MODEL_NAME = "Qwen/Qwen2.5-4B-Instruct"
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto", device_map="auto")
        print(f"Model loaded: {MODEL_NAME}")
    except Exception as e:
        print(f"Failed to load: {e}")
        USE_BACKEND = 'mock'

print(f"Backend: {USE_BACKEND}")


Model file not found: ./models/qwen2.5-4b-instruct-q4_k_m.gguf
Download from: https://huggingface.co/Qwen/Qwen2.5-4B-Instruct-GGUF
Falling back to MOCK MODE
Backend: mock


## 3. Parser Prompt Template

In [ ]:
PARSER_PROMPT = """You are a note parser. Extract structured data from the following note.

Note: "{raw_note}"
Today's date: "{today}"

Extract and return JSON only. No explanation. No markdown.

For money/ledger: {{"type": "ledger", "person": "", "amount": 0, "direction": "gave|received", "note": ""}}
For weight: {{"type": "weight", "person": "", "weight": 0.0, "note": ""}}
For expense: {{"type": "expense", "amount": 0, "description": ""}}
For todo: {{"type": "todo", "content": ""}}
If unclear: {{"type": "unknown", "content": ""}}

Rules:
- gave/give/given/lent/sent + person = ledger, direction: gave (user gave money to them)
- got/received/returned/paid back + person = ledger, direction: received (they gave money to user)
- gift keyword = expense, not ledger
- weight is a number associated with a person's name, usually below 150kg
- ledger amounts are usually 100 or more
- amount in k means thousands (5k = 5000), L means lakhs (1.5L = 150000)
- expense description is just the item name verbatim — do not classify it into a category
"""

def build_prompt(raw_note, today):
    return PARSER_PROMPT.format(raw_note=raw_note, today=today)

print("Prompt template defined.")
print("\nSample prompt:")
print(build_prompt("Moni sent 10k for groceries", "2026-05-02"))

## 4. Inference Function

In [ ]:

def extract_json(text):
    """Extract JSON from model output — handles markdown code blocks."""
    text = text.strip()
    # Remove markdown code blocks
    text = re.sub(r'```json\s*', '', text)
    text = re.sub(r'```\s*', '', text)
    text = text.strip()
    
    # Find first { ... } block
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1:
        return text[start:end+1]
    return text

def validate_parsed(result):
    """Validate Qwen output is well-formed."""
    t = result.get('type')
    if t == 'ledger':
        return (result.get('amount', 0) > 0 and
                result.get('direction') in ['gave', 'received'] and
                result.get('person', '').strip() != '')
    if t == 'expense':
        return result.get('amount', 0) > 0
    if t == 'weight':
        return result.get('weight', 0) > 0 and result.get('person', '').strip() != ''
    if t in ['todo', 'unknown']:
        return True
    return False

def qwen_parse(raw_note, today):
    """Run Qwen parser on a single note. Returns parsed dict or None."""
    prompt = build_prompt(raw_note, today)
    
    if USE_BACKEND == 'mock':
        return mock_parse(raw_note)
    
    try:
        if USE_BACKEND == 'llama_cpp':
            output = llm(prompt, max_tokens=150, temperature=0.0, stop=["\n\n"])
            raw_output = output['choices'][0]['text']
        
        elif USE_BACKEND == 'transformers':
            inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
            with torch.no_grad():
                output = model.generate(**inputs, max_new_tokens=150, temperature=0.01, do_sample=False)
            raw_output = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        json_str = extract_json(raw_output)
        result = json.loads(json_str)
        
        if validate_parsed(result):
            return result
        else:
            print(f"Validation failed for: {raw_note}")
            print(f"Output: {result}")
            return {'type': 'unknown', 'content': raw_note}
    
    except Exception as e:
        print(f"Parse error for '{raw_note}': {e}")
        return {'type': 'unknown', 'content': raw_note}

print("Inference function defined.")


## 5. Mock Parser (For Testing Without Model)

In [ ]:
def mock_parse(raw_note):
    """
    Mock Qwen responses for testing flow without the actual model.
    Replace this with real Qwen inference when model is available.
    """
    note_lower = raw_note.lower()
    
    mock_responses = {
        "moni sent 10k for groceries":      {'type': 'ledger',  'person': 'moni',   'amount': 10000, 'direction': 'received', 'note': 'for groceries'},
        "mani 500":                          {'type': 'expense', 'amount': 500,      'description': 'mani'},
        "iniyan iyar ku money 500":          {'type': 'ledger',  'person': 'iniyan', 'amount': 500,   'direction': 'gave',     'note': None},
        "got seetu money 23000":             {'type': 'ledger',  'person': 'self',   'amount': 23000, 'direction': 'received', 'note': 'seetu chit fund'},
        "paid electricity 3560":             {'type': 'expense', 'amount': 3560,     'description': 'electricity'},
        "advanced 1000 for mutton biryani":  {'type': 'expense', 'amount': 1000,     'description': 'mutton biryani advance'},
        "amma gave 6k for moni saree":       {'type': 'ledger',  'person': 'amma',   'amount': 6000,  'direction': 'received', 'note': 'for moni saree'},
    }
    
    for key, response in mock_responses.items():
        if key in note_lower:
            return response
    
    return {'type': 'unknown', 'content': raw_note}

print("Mock parser defined.")

## 6. Test Ambiguous Cases

In [ ]:
from datetime import datetime
TODAY = "2026-05-02T10:00:00"

ambiguous_cases = [
    ("Moni sent 10k for groceries",     'ledger',  {'direction': 'received', 'amount': 10000}),
    ("Mani 500",                        'expense', {'amount': 500}),
    ("iniyan iyar ku money 500",        'ledger',  {'direction': 'gave', 'amount': 500}),
    ("got seetu money 23000",           'ledger',  {'direction': 'received', 'amount': 23000}),
    ("paid electricity 3560",           'expense', {'amount': 3560}),
    ("advanced 1000 for mutton biryani",'expense', {'amount': 1000}),
    ("amma gave 6k for moni saree",     'ledger',  {'direction': 'received', 'amount': 6000}),
]

passed = 0
failed = 0

for raw, expected_type, expected_fields in ambiguous_cases:
    result = qwen_parse(raw, TODAY)
    
    actual_type = result.get('type', 'unknown')
    
    if actual_type == expected_type:
        field_ok = all(result.get(k) == v for k, v in expected_fields.items())
        if field_ok:
            print(f"✓ {raw}")
            passed += 1
        else:
            print(f"✗ FIELD FAIL | {raw}")
            print(f"  Expected: {expected_fields}")
            print(f"  Got:      { {k: result.get(k) for k in expected_fields} }")
            failed += 1
    else:
        print(f"✗ TYPE FAIL  | {raw}")
        print(f"  Expected: {expected_type}, Got: {actual_type}")
        failed += 1

print(f"\n{'='*50}")
pct = passed / len(ambiguous_cases) * 100
print(f"Score: {passed}/{len(ambiguous_cases)} = {pct:.1f}%")
print("EXIT CRITERIA: >= 80%")
print("PASS ✓" if pct >= 80 else "FAIL ✗ — iterate on prompt before Notebook 4")

## 7. Combined Flow: Pre-parser → Qwen Fallback

In [ ]:

import sys
sys.path.insert(0, '.')

# Import pre-parser from notebook 2 
# (copy parse_note function here or import from a .py file)
# For now, paste the key function inline:

def full_parse(raw_input, today=None):
    """
    Full parsing pipeline:
    1. Rule-based pre-parser
    2. If unknown → Qwen
    Returns list of parsed entries.
    """
    if today is None:
        today = datetime.now().strftime('%Y-%m-%dT%H:%M:%S')
    
    # Try rule-based first
    # (assumes parse_note from notebook 2 is available)
    try:
        results = parse_note(raw_input, today)
    except NameError:
        results = [{'type': 'unknown', 'content': raw_input, 'raw': raw_input}]
    
    # For any unknown entries, try Qwen
    final = []
    for r in results:
        if r['type'] == 'unknown':
            qwen_result = qwen_parse(r['content'], today)
            if qwen_result:
                qwen_result['raw'] = r['raw']
                final.append(qwen_result)
            else:
                final.append(r)
        else:
            final.append(r)
    
    return final

# Test combined flow
test_inputs = [
    "petrol 500",                        # rule-based handles
    "gave Maddy 5k",                     # rule-based handles
    "jeevi 62",                          # rule-based handles
    "Moni sent 10k for groceries",       # Qwen needed
    "update Amit about MCP",             # rule-based handles
]

print("Combined flow test:")
for raw in test_inputs:
    results = full_parse(raw, TODAY)
    for r in results:
        backend = "Qwen" if r.get('type') != 'unknown' and raw in ["Moni sent 10k for groceries"] else "rule-based"
        print(f"  {raw!r:40} → type={r['type']}, backend={backend}")
